# Decision Tree V1

For this first experiment, I used a Decision Tree with a maximum depth of 3 on both datasets. I started with 200 training samples and 200 validation samples to see how well it performs before trying different settings.

First, I imported the libraries and functions I'll need.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_task, split_train_val
from src.models.decision_tree import DecisionTreeClassifier

summary_rows = []


I started by testing the model on the image dataset.

In [2]:
X, y, _, _ = load_task("image")
X_train, X_val, y_train, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
image_X_train = X_train[:200]
image_y_train = y_train[:200]
image_X_val = X_val[:200]
image_y_val = y_val[:200]
assert image_X_train.shape == (200, 500)
assert image_X_val.shape == (200, 500)

image_clf = DecisionTreeClassifier(max_depth=3, min_samples=1)
print("Fitting image Decision Tree: 200 training samples, 500 features, max_depth=3")
start = perf_counter()
image_clf.fit(image_X_train, image_y_train)
image_fit_time = perf_counter() - start
image_training_accuracy = image_clf.score(image_X_train, image_y_train)
start = perf_counter()
image_predictions = image_clf.predict(image_X_val)
image_prediction_time = perf_counter() - start
image_validation_accuracy = sum(p == t for p, t in zip(image_predictions, image_y_val)) / len(image_y_val)
print(f"Image training accuracy: {image_training_accuracy:.2%}")
print(f"Image validation accuracy: {image_validation_accuracy:.2%}")
print(f"Fitting time: {image_fit_time:.6f} s")
print(f"Validation prediction time: {image_prediction_time:.6f} s")


Fitting image Decision Tree: 200 training samples, 500 features, max_depth=3
Image training accuracy: 14.50%
Image validation accuracy: 4.50%
Fitting time: 2.282449 s
Validation prediction time: 0.000720 s


I saved the image predictions and accuracy so I can compare them with the next experiments.

In [3]:
result_dir = project_root / "results" / "decision_tree"
result_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame({
    "validation_index": range(len(image_y_val)),
    "actual_label": image_y_val,
    "predicted_label": image_predictions,
}).to_csv(result_dir / "decision_tree_v1_image_validation.csv", index=False)
summary_rows.append({
    "dataset": "image",
    "version": "V1",
    "criterion": "entropy",
    "max_depth": image_clf.max_depth,
    "min_samples": image_clf.min_samples,
    "normalized": False,
    "training_samples": len(image_X_train),
    "validation_samples": len(image_X_val),
    "training_accuracy": image_training_accuracy,
    "validation_accuracy": image_validation_accuracy,
    "fit_time_seconds": image_fit_time,
    "prediction_time_seconds": image_prediction_time,
})
print("Saved 200 image validation predictions.")


Saved 200 image validation predictions.


Then I repeated the same test with the sentiment dataset.

In [4]:
X, y, _, _ = load_task("sentiment")
X_train, X_val, y_train, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
sentiment_X_train = X_train[:200]
sentiment_y_train = y_train[:200]
sentiment_X_val = X_val[:200]
sentiment_y_val = y_val[:200]
assert sentiment_X_train.shape == (200, 768)
assert sentiment_X_val.shape == (200, 768)

sentiment_clf = DecisionTreeClassifier(max_depth=3, min_samples=1)
print("Fitting sentiment Decision Tree: 200 training samples, 768 features, max_depth=3")
start = perf_counter()
sentiment_clf.fit(sentiment_X_train, sentiment_y_train)
sentiment_fit_time = perf_counter() - start
sentiment_training_accuracy = sentiment_clf.score(sentiment_X_train, sentiment_y_train)
start = perf_counter()
sentiment_predictions = sentiment_clf.predict(sentiment_X_val)
sentiment_prediction_time = perf_counter() - start
sentiment_validation_accuracy = sum(p == t for p, t in zip(sentiment_predictions, sentiment_y_val)) / len(sentiment_y_val)
print(f"Sentiment training accuracy: {sentiment_training_accuracy:.2%}")
print(f"Sentiment validation accuracy: {sentiment_validation_accuracy:.2%}")
print(f"Fitting time: {sentiment_fit_time:.6f} s")
print(f"Validation prediction time: {sentiment_prediction_time:.6f} s")


Fitting sentiment Decision Tree: 200 training samples, 768 features, max_depth=3
Sentiment training accuracy: 67.50%
Sentiment validation accuracy: 45.00%
Fitting time: 20.371912 s
Validation prediction time: 0.000639 s


I saved the sentiment results as well.

In [5]:
pd.DataFrame({
    "validation_index": range(len(sentiment_y_val)),
    "actual_label": sentiment_y_val,
    "predicted_label": sentiment_predictions,
}).to_csv(result_dir / "decision_tree_v1_sentiment_validation.csv", index=False)
summary_rows.append({
    "dataset": "sentiment",
    "version": "V1",
    "criterion": "entropy",
    "max_depth": sentiment_clf.max_depth,
    "min_samples": sentiment_clf.min_samples,
    "normalized": False,
    "training_samples": len(sentiment_X_train),
    "validation_samples": len(sentiment_X_val),
    "training_accuracy": sentiment_training_accuracy,
    "validation_accuracy": sentiment_validation_accuracy,
    "fit_time_seconds": sentiment_fit_time,
    "prediction_time_seconds": sentiment_prediction_time,
})
summary = pd.DataFrame(summary_rows)
summary.to_csv(result_dir / "decision_tree_v1_summary.csv", index=False)
print("Saved 200 sentiment validation predictions and both summary rows.")


Saved 200 sentiment validation predictions and both summary rows.


Finally, I compared the accuracy for both datasets.

In [6]:
print(summary[["dataset", "training_accuracy", "validation_accuracy", "fit_time_seconds", "prediction_time_seconds"]].to_string(index=False))


  dataset  training_accuracy  validation_accuracy  fit_time_seconds  prediction_time_seconds
    image              0.145                0.045          2.282449                 0.000720
sentiment              0.675                0.450         20.371912                 0.000639


In [7]:
tracker_path = result_dir / "kaggle_scores.csv"
columns = ["version", "image_validation_accuracy", "sentiment_validation_accuracy", "kaggle_public_score", "submitted_to_kaggle", "submission_filename", "notes"]
if tracker_path.exists():
    tracker = pd.read_csv(tracker_path, dtype=str, keep_default_na=False)
else:
    tracker = pd.DataFrame(columns=columns)
assert tracker["version"].is_unique
v1_row = {
    "version": "V1",
    "image_validation_accuracy": image_validation_accuracy,
    "sentiment_validation_accuracy": sentiment_validation_accuracy,
    "kaggle_public_score": "",
    "submitted_to_kaggle": False,
    "submission_filename": "",
    "notes": "max_depth=3, min_samples=1; 200 training examples per dataset; entropy/information gain.",
}
matching = tracker["version"] == "V1"
if matching.any():
    for column in ("image_validation_accuracy", "sentiment_validation_accuracy", "submission_filename", "notes"):
        tracker.loc[matching, column] = v1_row[column]
else:
    tracker = pd.concat([tracker, pd.DataFrame([v1_row])], ignore_index=True)
assert tracker["version"].is_unique
tracker.to_csv(tracker_path, index=False)
print(tracker.to_string(index=False))


version  image_validation_accuracy  sentiment_validation_accuracy kaggle_public_score submitted_to_kaggle submission_filename                                                                                    notes
     V1                      0.045                           0.45                                   False                     max_depth=3, min_samples=1; 200 training examples per dataset; entropy/information gain.
